# Benchmark an LLM on real devices with TinyEdge

[TinyEdge](https://tinyedge.ai) runs GGUF models on real phones and tablets and reports decode tok/s, time-to-first-token, RAM, and perplexity. This notebook:

1. Benchmarks a model on your devices.
2. Runs `optimize=True` to compare quantizations and see which one to ship per device.

**Before you run:** a free [tinyedge.ai](https://tinyedge.ai) account (comes with demo credit; this notebook costs about a dollar), your API key in the first cell, at least one device paired in the TinyEdge Runner app with "Available for benchmarks" on, and internet enabled (Kaggle: right sidebar).

In [ ]:
%pip -q install "tinyedge[hf]>=0.4.3"

In [ ]:
import tinyedge

# Paste your API key from tinyedge.ai (New benchmark > Your API key):
client = tinyedge.TinyEdge(api_key="tinyedge_sk_REPLACE_ME")

DEVICES = client.devices(online=True)   # devices online right now
print("benchmarking on:", DEVICES)

## Benchmark a model

Pass an `hf:<owner>/<repo>/<file.gguf>` reference, or a local path. Swap in any GGUF (Qwen2.5, Gemma-2, Phi-3, your own).

In [ ]:
MODEL = "hf:bartowski/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct-Q4_K_M.gguf"
# dataset="wikitext" below fetches a WikiText sample automatically (for perplexity) —
# no setup cell needed. Pass your own folder/archive path instead to eval on your data.

In [ ]:
# one row per device: tok/s, time-to-first-token, RAM, perplexity
client.benchmark(MODEL, devices=DEVICES, dataset="wikitext")

## Compare quantizations (optimize=True)

`optimize=True` builds the whole quant ladder on TinyEdge's cloud GPUs — the importance matrix and per-rung perplexity run on GPU, and your notebook never downloads the multi-GB f16. It drops variants that hurt quality, then benchmarks the rest on your devices as one sweep, reusing the corpus for perplexity. The report marks the **Pareto-optimal** variants (the ones worth shipping — you can't get smaller, faster, *and* higher-quality than them), and `plot_pareto()` graphs the size↔quality trade-off.

Pass the HuggingFace repo (not a local file) so the build runs in the cloud:

In [ ]:
report = client.benchmark(
    "hf:bartowski/Llama-3.2-1B-Instruct-GGUF",   # repo ref -> cloud-GPU build
    devices=DEVICES, dataset="wikitext", optimize=True,
)
print(report.summary())          # variants on the Pareto frontier are starred
print("full report:", report.sweep_url)

report.plot_pareto()             # size-vs-quality, frontier highlighted